In [2]:
import requests
import pandas as pd


In [ ]:
# Configurações do SonarQube
SONAR_TOKEN = os.getenv('SONAR_TOKEN')
BASE_URL_SONAR = "http://localhost:9000"

In [4]:
# Cabeçalhos para autenticação
headers = {
    "Authorization": f"Bearer {SONAR_TOKEN}"
}

In [17]:
# Obter a lista de projetos no SonarQube
response_projects = requests.get(f"{BASE_URL_SONAR}/api/projects/search", headers=headers)

if response_projects.status_code == 200:
    projects_data = response_projects.json()
    projects = [p["key"] for p in projects_data.get("components", [])]

    results = []

    for project_key in projects:
        print(f"🔍 Analisando projeto: {project_key}")

        # Obter Número Total de Arquivos e Linhas de Código
        params_metrics = {
            "component": project_key,
            "metricKeys": "files,ncloc"
        }
        response_metrics = requests.get(f"{BASE_URL_SONAR}/api/measures/component", headers=headers, params=params_metrics)

        # Obter Número de Arquivos com Issues CRITICAL ou BLOCKER
        params_critical_issues = {
            "componentKeys": project_key,
            "severities": "CRITICAL,BLOCKER",
            "statuses": "OPEN",
            "facets": "files",
            "ps": 500
        }
        response_critical_issues = requests.get(f"{BASE_URL_SONAR}/api/issues/search", headers=headers, params=params_critical_issues)

        # Obter Número de Issues Abertas e Confirmadas do Tipo BUG
        params_bug_issues = {
            "componentKeys": project_key,
            "statuses": "OPEN,CONFIRMED",
            "types": "BUG",
            "ps": 500
        }
        response_bug_issues = requests.get(f"{BASE_URL_SONAR}/api/issues/search", headers=headers, params=params_bug_issues)

        if (
            response_metrics.status_code == 200
            and response_critical_issues.status_code == 200
            and response_bug_issues.status_code == 200
        ):
            metrics_data = response_metrics.json().get("component", {}).get("measures", [])
            total_files = int(next((m["value"] for m in metrics_data if m["metric"] == "files"), 0))
            total_lines_of_code = int(next((m["value"] for m in metrics_data if m["metric"] == "ncloc"), 0))

            critical_issues_data = response_critical_issues.json()
            affected_files = set(issue["component"] for issue in critical_issues_data.get("issues", []))

            total_bug_issues = response_bug_issues.json().get("total", 0)

            # Cálculo das métricas
            bug_density_loc = 1 - (total_bug_issues / total_lines_of_code) if total_lines_of_code > 0 else 0
            critical_blocker_ratio = len(affected_files) / total_files if total_files > 0 else 0

            results.append({
                "Projeto": project_key,
                "Total de Arquivos": total_files,
                "Linhas de Código": total_lines_of_code,
                "Arquivos com Issues CRITICAL/BLOCKER": len(affected_files),
                "Issues Abertas e Confirmadas (BUGs)": total_bug_issues,
                "Bug Density LOC": bug_density_loc,
                "Proporção de Arquivos com CRITICAL/BLOCKER": critical_blocker_ratio
            })

        else:
            print(f"⚠️ Erro ao buscar métricas para {project_key}")

🔍 Analisando projeto: Core-develop
🔍 Analisando projeto: Core-remotes-origin-bug-008
🔍 Analisando projeto: Core-remotes-origin-bug-019
🔍 Analisando projeto: Core-remotes-origin-christian-bug-018
🔍 Analisando projeto: Core-remotes-origin-develop
🔍 Analisando projeto: Core-remotes-origin-fix-sonar
🔍 Analisando projeto: Core-remotes-origin-US010-Threshould
🔍 Analisando projeto: Measure-develop
🔍 Analisando projeto: Measure-remotes-origin-BUG012
🔍 Analisando projeto: Measure-remotes-origin-BUG013
🔍 Analisando projeto: Measure-remotes-origin-BUG023
🔍 Analisando projeto: Measure-remotes-origin-BUG034
🔍 Analisando projeto: Measure-remotes-origin-BUG040
🔍 Analisando projeto: Measure-remotes-origin-cf-fix-BUG004
🔍 Analisando projeto: Measure-remotes-origin-christian-ENH008
🔍 Analisando projeto: Measure-remotes-origin-develop
🔍 Analisando projeto: Measure-remotes-origin-ENH004
🔍 Analisando projeto: Measure-remotes-origin-ENH009
🔍 Analisando projeto: Measure-remotes-origin-feat-ENH005
🔍 Analisand

In [18]:
# Criar DataFrame Pandas
df = pd.DataFrame(results)

In [19]:
# Exibir os dados no terminal
df

,Projeto,Total de Arquivos,Linhas de Código,Arquivos com Issues CRITICAL/BLOCKER,Issues Abertas e Confirmadas (BUGs),Bug Density LOC,Proporção de Arquivos com CRITICAL/BLOCKER
0,Core-develop,16,1392,2,0,1.000000,0.125000
1,Core-remotes-origin-bug-008,15,1122,1,0,1.000000,0.066667
2,Core-remotes-origin-bug-019,15,1130,1,0,1.000000,0.066667
3,Core-remotes-origin-christian-bug-018,15,1122,1,0,1.000000,0.066667
4,Core-remotes-origin-develop,16,1392,2,0,1.000000,0.125000
5,Core-remotes-origin-fix-sonar,15,1122,1,0,1.000000,0.066667
6,Core-remotes-origin-US010-Threshould,16,1392,2,0,1.000000,0.125000
7,Measure-develop,16,1392,2,0,1.000000,0.125000
8,Measure-remotes-origin-BUG012,312,15606,5,3,0.999808,0.016026
9,Measure-remotes-origin-BUG013,288,16149,10,5,0.999690,0.034722
